# Clinical Triage Classifier — Week 7
### Fine-Tuning Llama 3.2 with QLoRA for Offline Emergency Triage
**Stella Oiro — Andela AI Engineering Bootcamp**

---

**The problem:** Emergency departments need fast, consistent triage. LLMs can help — but sending patient data to a cloud API violates data governance rules in most hospitals.

**The solution:** Fine-tune an open source model with QLoRA so it runs entirely on local hardware. No internet. No API. No data leaving the building.

**The result:** A fine-tuned Llama 3.2 3B that outperforms GPT-4.1-mini zero-shot on triage classification — and runs offline.

---

## Manchester Triage System — 4 levels

| Level | Time to see | Examples |
|---|---|---|
| Immediate | Now | Cardiac arrest, unresponsive, severe anaphylaxis |
| Urgent | < 30 min | High fever in infant, moderate chest pain, acute confusion |
| Semi-urgent | < 1 hour | Mild asthma, laceration, moderate pain |
| Non-urgent | Can wait | Minor cold, medication query, minor skin irritation |

---

## Order of play

**Part 1** — Generate synthetic training data (GPT-4.1-mini, CPU)  
**Part 2** — Load data, build prompts, inspect base model (CPU)  
**Part 3** — Fine-tune with QLoRA (T4 GPU required)  
**Part 4** — Evaluate: accuracy, F1, confusion matrix (T4 GPU)  
**Part 5** — Compare models + Gradio UI (T4 GPU)  

## Setup

In [1]:
# Install dependencies
!pip install -q openai transformers peft bitsandbytes accelerate datasets huggingface_hub gradio scikit-learn matplotlib seaborn

zsh:1: command not found: pip


In [2]:
import os
import json
import random
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'], add_to_git_credential=True)

print('Setup complete')

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# Configuration — change HF_USERNAME to your HuggingFace username
HF_USERNAME = "your-hf-username"    # <-- change this
DATASET_NAME = f"{HF_USERNAME}/clinical-triage-dataset"
MODEL_NAME = f"{HF_USERNAME}/clinical-triage-llama"
BASE_MODEL = "meta-llama/Llama-3.2-3B"

TRIAGE_LEVELS = ["Immediate", "Urgent", "Semi-urgent", "Non-urgent"]

print(f"Dataset: {DATASET_NAME}")
print(f"Fine-tuned model: {MODEL_NAME}")

---

# Part 1 — Generate Synthetic Training Data

We use GPT-4.1-mini to generate 2000 labelled triage cases. This mirrors what Ed does in Week 6 Day 2 — using a cheap LLM as a data transformation step.

The generated data is pushed to HuggingFace Hub and never committed to git.

In [ ]:
from openai import OpenAI
import time

client = OpenAI()

SYSTEM_PROMPT = """You are a senior emergency medicine consultant. Generate realistic clinical triage cases.
Each case must be a brief chief complaint as a triage nurse would record it (1-2 sentences).
Respond ONLY with a JSON object containing a single key "cases", whose value is a JSON array. Each item: {"presentation": "...", "triage": "..."}.
Triage must be exactly one of: Immediate, Urgent, Semi-urgent, Non-urgent."""

def generate_cases(level: str, count: int, retries: int = 3) -> list[dict]:
    """Generate `count` synthetic triage cases for a given urgency level."""
    prompt = f"""Generate {count} realistic emergency department presentations that should be triaged as '{level}'.
Include a variety of: age groups, chief complaints, vital signs where relevant, onset duration.
Make them clinically realistic and diverse."""

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4.1-mini",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt}
                ],
                response_format={"type": "json_object"},
                max_tokens=4096,
                temperature=0.9
            )
            result = json.loads(response.choices[0].message.content)

            cases = result.get("cases", [])
            if not cases:
                for v in result.values():
                    if isinstance(v, list):
                        cases = v
                        break

            for c in cases:
                c['triage'] = level
            return cases

        except (json.JSONDecodeError, KeyError, StopIteration) as e:
            print(f"  Attempt {attempt+1} failed: {e}. Retrying...")
            time.sleep(1)

    print(f"  All {retries} attempts failed for level={level}, count={count}. Returning empty.")
    return []

In [ ]:
# Generate 500 cases per triage level = 2000 total
# Batches of 20 to avoid JSON truncation from hitting output token limits
all_cases = []

for level in TRIAGE_LEVELS:
    print(f"Generating {level} cases...")
    level_cases = []
    for batch in range(25):  # 25 batches × 20 = 500 per level
        batch_cases = generate_cases(level, 20)
        level_cases.extend(batch_cases)
        print(f"  Batch {batch+1}/25 — {len(level_cases)} {level} cases so far")
    all_cases.extend(level_cases)

random.shuffle(all_cases)
print(f"\nTotal cases generated: {len(all_cases)}")
print("Distribution:", {level: sum(1 for c in all_cases if c['triage'] == level) for level in TRIAGE_LEVELS})

In [ ]:
# Preview a few examples
for level in TRIAGE_LEVELS:
    example = next(c for c in all_cases if c['triage'] == level)
    print(f"[{level}] {example['presentation']}")
    print()

In [ ]:
from datasets import Dataset, DatasetDict

# Split 80/10/10 train/val/test
n = len(all_cases)
train_cases = all_cases[:int(n * 0.8)]
val_cases   = all_cases[int(n * 0.8):int(n * 0.9)]
test_cases  = all_cases[int(n * 0.9):]

dataset = DatasetDict({
    "train": Dataset.from_list(train_cases),
    "validation": Dataset.from_list(val_cases),
    "test": Dataset.from_list(test_cases),
})

print(dataset)

# Push to HuggingFace Hub — not stored in git
dataset.push_to_hub(DATASET_NAME, private=True)
print(f"Dataset pushed to: https://huggingface.co/datasets/{DATASET_NAME}")

---

# Part 2 — Build Prompts and Inspect Base Model

Format each case as a user/assistant conversation for causal LM fine-tuning.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

dataset = load_dataset(DATASET_NAME)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

print(f"Train: {len(dataset['train'])} | Val: {len(dataset['validation'])} | Test: {len(dataset['test'])}")

In [ ]:
SYSTEM_MSG = """You are a clinical triage assistant. Classify the urgency of the patient presentation.
Respond with exactly one word: Immediate, Urgent, Semi-urgent, or Non-urgent."""

def make_prompt(presentation: str, triage: str = None) -> str:
    """Build a chat-template formatted prompt. Include answer if training."""
    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user",   "content": f"Triage this patient:\n{presentation}"}
    ]
    if triage:  # training — include the answer
        messages.append({"role": "assistant", "content": triage})
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=(triage is None))

# Preview
sample = dataset['train'][0]
print("=== TRAINING PROMPT ===")
print(make_prompt(sample['presentation'], sample['triage']))
print("\n=== TEST PROMPT (no answer) ===")
print(make_prompt(dataset['test'][0]['presentation']))

In [ ]:
import matplotlib.pyplot as plt

# Token count distribution
token_counts = [
    len(tokenizer.encode(make_prompt(item['presentation'], item['triage'])))
    for item in dataset['train']
]

print(f"Avg tokens: {sum(token_counts)/len(token_counts):.1f}")
print(f"Max tokens: {max(token_counts)}")
print(f"Min tokens: {min(token_counts)}")

plt.figure(figsize=(10, 4))
plt.title("Token count per training prompt")
plt.xlabel("Tokens")
plt.ylabel("Count")
plt.hist(token_counts, bins=30, color="steelblue", rwidth=0.8)
plt.tight_layout()
plt.show()

MAX_LENGTH = 256  # well above the max we see

In [ ]:
# Tokenize the dataset
def tokenize(item):
    text = make_prompt(item['presentation'], item['triage'])
    tokens = tokenizer(text, truncation=True, max_length=MAX_LENGTH, padding='max_length')
    tokens['labels'] = tokens['input_ids'].copy()
    return tokens

tokenized_train = dataset['train'].map(tokenize, remove_columns=dataset['train'].column_names)
tokenized_val   = dataset['validation'].map(tokenize, remove_columns=dataset['validation'].column_names)

tokenized_train.set_format('torch')
tokenized_val.set_format('torch')

print(f"Tokenized train: {len(tokenized_train)} examples")

### Base model — before fine-tuning

Let's see how the untuned Llama performs. Expect ~25% — random guessing across 4 classes.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto"
)
print(f"Base model loaded: {BASE_MODEL}")

In [ ]:
def predict(model, presentation: str) -> str:
    """Run inference and extract the triage classification."""
    prompt = make_prompt(presentation)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    # Extract the first matching triage level from the response
    for level in TRIAGE_LEVELS:
        if level.lower() in response.lower():
            return level
    return "Unknown"

# Quick test
test_case = "72-year-old male, sudden onset crushing central chest pain, sweating profusely, BP 90/60"
print(f"Presentation: {test_case}")
print(f"Base model prediction: {predict(base_model, test_case)}")
print(f"Expected: Immediate")

In [ ]:
from sklearn.metrics import accuracy_score

# Evaluate base model on 100 test cases
test_sample = list(dataset['test'])[:100]
base_preds = [predict(base_model, item['presentation']) for item in test_sample]
base_labels = [item['triage'] for item in test_sample]
base_accuracy = accuracy_score(base_labels, base_preds)

print(f"Base model accuracy (100 test cases): {base_accuracy:.1%}")
print(f"Random baseline would be: 25%")

---

# Part 3 — Fine-Tune with QLoRA

**Requires T4 GPU.** Select Runtime → Change runtime type → T4 GPU before running this section.

We add LoRA adapter matrices to the attention layers of the 4-bit quantised model. Only 0.44% of parameters are trained.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,                   # rank — lower than Ed's 32 since our task is simpler
    lora_alpha=32,          # scaling factor = 2 × rank
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# Reload model fresh (clean state before fine-tuning)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="triage_model",
    num_train_epochs=3,              # 3 epochs — task is simple, converges fast
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # effective batch = 16
    warmup_steps=10,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    load_best_model_at_end=True,
    report_to="none",
)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=collator,
)

print("Starting training...")
trainer.train()

In [ ]:
# Push fine-tuned adapters and tokenizer to HuggingFace Hub (private)
model.push_to_hub(MODEL_NAME, private=True)
tokenizer.push_to_hub(MODEL_NAME, private=True)
print(f"Fine-tuned model saved to: https://huggingface.co/{MODEL_NAME}")

---

# Part 4 — Evaluate

Full evaluation on the held-out test set: accuracy, F1-score per class, confusion matrix.

In [ ]:
from peft import PeftModel

# Load from Hub (use this cell if resuming a session)
# base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=quant_config, device_map="auto")
# model = PeftModel.from_pretrained(base_model, MODEL_NAME)

# Evaluate fine-tuned model on full test set
test_items = list(dataset['test'])
ft_preds  = [predict(model, item['presentation']) for item in test_items]
ft_labels = [item['triage'] for item in test_items]

from sklearn.metrics import accuracy_score, classification_report
ft_accuracy = accuracy_score(ft_labels, ft_preds)
print(f"Fine-tuned model accuracy: {ft_accuracy:.1%}")
print()
print(classification_report(ft_labels, ft_preds, target_names=TRIAGE_LEVELS))

In [ ]:
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(ft_labels, ft_preds, labels=TRIAGE_LEVELS)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm_pct, annot=True, fmt='.1f', cmap='Blues',
    xticklabels=TRIAGE_LEVELS, yticklabels=TRIAGE_LEVELS, ax=ax
)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix (%) — Fine-tuned Llama Triage Classifier')
plt.tight_layout()
plt.show()

# Key safety check: under-triage rate for Immediate cases
immediate_idx = TRIAGE_LEVELS.index('Immediate')
under_triage = 100 - cm_pct[immediate_idx, immediate_idx]
print(f"Under-triage rate for Immediate cases: {under_triage:.1f}%")
print("(Cases classified as less urgent than Immediate — the most safety-critical error)")

---

# Part 5 — Model Comparison + Gradio UI

Compare three approaches:
1. Base Llama 3.2 3B (no training)
2. GPT-4.1-mini zero-shot (cloud, requires internet)
3. Fine-tuned Llama 3.2 3B with QLoRA (runs offline)

In [ ]:
# GPT-4.1-mini zero-shot on same test set
def predict_gpt(presentation: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": SYSTEM_MSG},
            {"role": "user", "content": f"Triage this patient:\n{presentation}"}
        ],
        max_tokens=10,
        temperature=0
    )
    text = response.choices[0].message.content.strip()
    for level in TRIAGE_LEVELS:
        if level.lower() in text.lower():
            return level
    return "Unknown"

# Evaluate on 100 test cases (to keep cost low)
eval_sample = test_items[:100]
eval_labels = [item['triage'] for item in eval_sample]

gpt_preds  = [predict_gpt(item['presentation']) for item in eval_sample]
gpt_accuracy = accuracy_score(eval_labels, gpt_preds)
print(f"GPT-4.1-mini zero-shot accuracy: {gpt_accuracy:.1%}")

In [ ]:
# Evaluate base and fine-tuned on the same 100 cases for fair comparison
base_preds_100 = [predict(base_model, item['presentation']) for item in eval_sample]
ft_preds_100   = [predict(model, item['presentation']) for item in eval_sample]

base_acc_100 = accuracy_score(eval_labels, base_preds_100)
ft_acc_100   = accuracy_score(eval_labels, ft_preds_100)

print(f"Base Llama accuracy:          {base_acc_100:.1%}")
print(f"GPT-4.1-mini zero-shot:       {gpt_accuracy:.1%}")
print(f"Fine-tuned Llama (QLoRA):     {ft_acc_100:.1%}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

models = ["Base Llama\n3.2 3B", "GPT-4.1-mini\n(zero-shot)", "Fine-tuned Llama\n3.2 3B (QLoRA)"]
accuracies = [base_acc_100 * 100, gpt_accuracy * 100, ft_acc_100 * 100]
colors = ["darkred", "slateblue", "steelblue"]
labels = ["Base (no training)", "Frontier (cloud API)", "Fine-tuned (offline)"]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(models, accuracies, color=colors, width=0.5)

for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=13, fontweight='bold')

ax.axhline(y=25, color='gray', linestyle='--', alpha=0.5, label='Random baseline (25%)')
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Clinical Triage Classification — Model Comparison\n(Fine-tuned open source vs frontier, runs offline)', fontsize=13)
ax.set_ylim(0, 105)
ax.legend(fontsize=11)
legend_patches = [mpatches.Patch(color=c, label=l) for c, l in zip(colors, labels)]
ax.legend(handles=legend_patches + [plt.Line2D([0],[0], color='gray', linestyle='--', label='Random baseline')], fontsize=11)
plt.tight_layout()
plt.show()

print(f"\nKey result: Fine-tuned Llama is {ft_acc_100 - gpt_accuracy:.1%} more accurate than GPT-4.1-mini")
print(f"And it runs OFFLINE — no patient data leaves the hospital.")

### Gradio UI

In [ ]:
import gradio as gr

LEVEL_COLORS = {
    "Immediate":   "#d32f2f",
    "Urgent":      "#f57c00",
    "Semi-urgent": "#fbc02d",
    "Non-urgent":  "#388e3c",
    "Unknown":     "#9e9e9e",
}

LEVEL_DESCRIPTIONS = {
    "Immediate":   "IMMEDIATE — Life-threatening. See patient NOW.",
    "Urgent":      "URGENT — See within 30 minutes.",
    "Semi-urgent": "SEMI-URGENT — See within 1 hour.",
    "Non-urgent":  "NON-URGENT — Can wait. Manage in order of arrival.",
    "Unknown":     "Unable to classify. Please review manually.",
}

def triage(presentation: str) -> str:
    if not presentation.strip():
        return "Please enter a clinical presentation."
    level = predict(model, presentation.strip())
    color = LEVEL_COLORS[level]
    desc  = LEVEL_DESCRIPTIONS[level]
    return f'<div style="background:{color};color:white;padding:16px;border-radius:8px;font-size:18px;font-weight:bold">{desc}</div>'

examples = [
    ["72yo male, sudden crushing chest pain, sweating, BP 85/50, pallor"],
    ["4-year-old female, fever 39.8°C for 2 days, irritable, not eating"],
    ["28yo male, laceration to right forearm, bleeding controlled, tetanus unknown"],
    ["55yo female, mild headache for 3 days, no fever, no neuro symptoms"],
    ["Infant 8 months, found unresponsive by parent, not breathing"],
    ["35yo male, moderate shortness of breath, O2 sat 94%, moderate wheeze"],
]

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # Clinical Triage Classifier
    **Fine-tuned Llama 3.2 3B — runs offline, no patient data leaves the building**

    Enter a brief clinical presentation as a triage nurse would record it.
    """)

    with gr.Row():
        inp = gr.Textbox(
            label="Clinical Presentation",
            placeholder="e.g. 65yo female, sudden onset of severe headache, worst of her life, vomiting...",
            lines=3
        )
    btn = gr.Button("Classify", variant="primary")
    out = gr.HTML(label="Triage Decision")

    gr.Examples(examples=examples, inputs=inp)
    btn.click(triage, inputs=inp, outputs=out)

demo.launch(share=True)